# Diagnostic: does the real data contain a length-scale signal?

The Laplace recovery notebook found that `length_scale` is only weakly identified — the objective
plateaus for `ls` above ~1 (the feature-cloud diameter is ~0.78, so `ls>1` is already "smoother than
the data's spatial extent"). Open question: is that a synthetic-data artifact, or will real data show
the same plateau?

`length_scale` is identifiable **if and only if** predicted/observed prevalence *falls off with
distance from the trained region*. A large `ls` over-generalizes (predicts high coherence far from
training); it only pays a likelihood cost if there are test features that are far from training AND
rated low. So two model-free checks on the REAL ratings settle it:

1. **Empirical prevalence range per condition** — compare to the synthetic ranges
   (diet 0.24-0.35, personality 0.55-0.58, physical 0.56-0.73, heterogeneous 0.40-0.43). A wider
   real range means more coherence contrast (rougher field => smaller true `ls` => `ls` recoverable);
   a similar/narrower range means the plateau will persist on real data.
2. **Distance-to-nearest-trained-feature vs mean rating** — the decisive test. A NEGATIVE correlation
   = prevalence declines with distance = the `ls` signal exists in real data. A FLAT scatter = it
   doesn't, and `ls` should be fixed / tightly-prior'd regardless of what synthetic said.

No model, no VBMC — just the raw ratings and the real embeddings.

In [ ]:
import sys, csv, pickle as pkl
sys.path.insert(0, ".")
import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

# ---- feature geometry (same as the recovery notebooks) ----
with open('../features/set2_features_dataframe.pkl', 'rb') as f:
    df = pkl.load(f)
train_df = df[df.split == 'train']
feat_idx = df.set_index('feature')

CONDITIONS  = ['diet', 'personality', 'physical', 'heterogeneous']
CAT_OF_COND = {'diet':'diet_preferences', 'personality':'personality_behaviors', 'physical':'physical'}
CAT_SHORT   = {'diet_preferences':'diet', 'personality_behaviors':'personality', 'physical':'physical'}

x_train_cond = {}
for c in CONDITIONS:
    sub = train_df[train_df.in_heterogenous] if c == 'heterogeneous' else train_df[train_df.category == CAT_OF_COND[c]]
    x_train_cond[c] = np.array(sub[['x_2d', 'y_2d']].values)

CSV_TO_FEATURE = {
    'diet_can_eat_spicy_1':'can eat spicy food','diet_breakfast_late_1':'eat breakfast very late',
    'diet_five_meals_day_1':'eat five meals a day','diet_like_juice_pulp_1':'like juice with pulp',
    'diet_pepper_on_all_1':'put pepper on all their foods','pers_cry_easily_1':'cry easily',
    'pers_collect_rocks_1':'like to collect rocks','pers_like_to_dance_1':'like to dance',
    'pers_like_highfive_1':'like to give high-fives','pers_read_books_1':'like to read books',
    'phys_can_roll_tongue_1':'can roll their tongue','phys_can_snap_toes_1':'can snap with their toes',
    'phys_can_wiggle_ears_1':'can wiggle their ears','phys_cold_hands_feet_1':'have cold hands and feet',
    'phys_snore_sleep_1':'snore when they sleep'}
TEST_CSV_COLS      = list(CSV_TO_FEATURE.keys())
test_feature_names = list(CSV_TO_FEATURE.values())
x_test     = np.array([feat_idx.loc[n, ['x_2d','y_2d']].values for n in test_feature_names], dtype=float)  # (J,2)
test_trait = [CAT_SHORT[feat_idx.loc[n, 'category']] for n in test_feature_names]
J = x_test.shape[0]

# cloud diameter, for context on what "large ls" means
all_xy = np.array(df[['x_2d','y_2d']].values, dtype=float)
cloud_diam = np.sqrt(((all_xy[:,None,:]-all_xy[None,:,:])**2).sum(-1)).max()
print(f"J={J} test features; feature-cloud diameter = {cloud_diam:.3f}")

In [ ]:
# ---- real ratings, stratified by condition (from study9.csv) ----
with open('../../data/study9.csv') as f:
    rows = list(csv.reader(f))
hdr = rows[0]; col_idx = [hdr.index(c) for c in TEST_CSV_COLS]; cond_i = hdr.index('condition')

R, cond = [], []
for r in rows[3:]:
    try:
        R.append([int(r[i]) / 100.0 for i in col_idx])
    except (ValueError, IndexError):
        continue
    cond.append(r[cond_i])
R = np.array(R); cond = np.array(cond)

responses_cond = {c: R[cond == c] for c in CONDITIONS}
for c in CONDITIONS:
    print(f"  {c:<14}: {responses_cond[c].shape[0]} participants x {responses_cond[c].shape[1]} features")

## Check 1: empirical prevalence range per condition (real vs synthetic)

Mean rating per test feature, per condition. Compare the real spread against the synthetic `pz1`
ranges the recovery notebook drew from `ls=2.1`. Wider real range => more contrast => `ls` more
identifiable; similar/narrower => plateau persists.

In [ ]:
SYNTH_RANGE = {'diet': (0.24, 0.35), 'personality': (0.55, 0.58),
               'physical': (0.56, 0.73), 'heterogeneous': (0.40, 0.43)}

mean_rating_cond = {c: responses_cond[c].mean(axis=0) for c in CONDITIONS}   # (J,) each

print(f"{'condition':<14}{'real range':>18}{'real spread':>13}{'synth range':>18}{'synth spread':>14}")
for c in CONDITIONS:
    m = mean_rating_cond[c]
    lo, hi = m.min(), m.max()
    slo, shi = SYNTH_RANGE[c]
    print(f"{c:<14}[{lo:.2f}, {hi:.2f}]{'':>6}{hi-lo:>11.2f}"
          f"   [{slo:.2f}, {shi:.2f}]{'':>4}{shi-slo:>11.2f}")

fig, axes = plt.subplots(1, 4, figsize=(18, 3), sharey=True)
for ax, c in zip(axes, CONDITIONS):
    m = mean_rating_cond[c]
    ax.bar(range(J), m, color='steelblue', alpha=0.7)
    ax.axhspan(*SYNTH_RANGE[c], color='tomato', alpha=0.2, label='synthetic pz1 range')
    ax.set_xticks(range(J)); ax.set_xticklabels(test_feature_names, rotation=45, ha='right', fontsize=6)
    ax.set_title(f'{c}  (real spread {m.max()-m.min():.2f})')
axes[0].set_ylabel('mean rating'); axes[0].legend(fontsize=7)
fig.suptitle('Real empirical prevalence per condition (bars) vs synthetic pz1 range (red band)')
plt.tight_layout(); plt.show()

## Check 2 (decisive): does prevalence fall off with distance from the trained region?

For each test feature, distance to its own condition's nearest trained feature (in the 2D
embedding). Correlate with mean rating. A **negative** correlation = prevalence declines with
distance = the identifying signal for `ls` exists in real data. A **flat** scatter = `ls` is
unidentifiable from this design regardless of the synthetic result.

In [ ]:
def nearest_train_dist(xt, x_train):
    d = np.sqrt(((x_train[None, :, :] - xt[:, None, :]) ** 2).sum(-1))   # (J, n_train)
    return d.min(axis=1)                                                  # (J,)

dist_cond = {c: nearest_train_dist(x_test, x_train_cond[c]) for c in CONDITIONS}

# pooled across conditions (z-score mean rating within condition so levels don't dominate)
all_dist, all_rating_z, all_rating_raw, all_c = [], [], [], []
for c in CONDITIONS:
    m = mean_rating_cond[c]
    mz = (m - m.mean()) / (m.std() + 1e-12)
    all_dist.append(dist_cond[c]); all_rating_z.append(mz); all_rating_raw.append(m)
    all_c += [c] * J
all_dist = np.concatenate(all_dist); all_rating_z = np.concatenate(all_rating_z); all_rating_raw = np.concatenate(all_rating_raw)

print("per-condition  distance vs mean-rating correlation:")
for c in CONDITIONS:
    m = mean_rating_cond[c]; d = dist_cond[c]
    if np.std(d) < 1e-9:
        print(f"  {c:<14}: (no distance variation)")
    else:
        r, p = pearsonr(d, m)
        print(f"  {c:<14}: pearson r={r:+.3f}  (p={p:.3f})  dist range [{d.min():.2f}, {d.max():.2f}]")

rp, pp = pearsonr(all_dist, all_rating_z)
rs, ps = spearmanr(all_dist, all_rating_z)
print(f"\nPOOLED (within-condition z-scored):  pearson r={rp:+.3f} (p={pp:.3f})   spearman r={rs:+.3f} (p={ps:.3f})")
print("negative r => prevalence declines with distance => length_scale IS identifiable in real data")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {'diet':'tab:blue','personality':'tab:green','physical':'tab:orange','heterogeneous':'tab:purple'}
for c in CONDITIONS:
    axes[0].scatter(dist_cond[c], mean_rating_cond[c], s=45, color=colors[c], label=c, alpha=0.8)
axes[0].set_xlabel('distance to nearest trained feature'); axes[0].set_ylabel('mean rating (raw)')
axes[0].set_title('Raw mean rating vs distance'); axes[0].legend(fontsize=8)

axes[1].scatter(all_dist, all_rating_z, s=45, c=[colors[c] for c in all_c], alpha=0.8)
b1, b0 = np.polyfit(all_dist, all_rating_z, 1)
xs = np.linspace(all_dist.min(), all_dist.max(), 50)
axes[1].plot(xs, b1*xs + b0, 'k--', label=f'slope={b1:+.2f}, r={rp:+.2f}')
axes[1].set_xlabel('distance to nearest trained feature'); axes[1].set_ylabel('mean rating (within-cond z)')
axes[1].set_title('Pooled: z-scored rating vs distance'); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

## Verdict

- **Check 1 wider than synthetic** *and* **Check 2 clearly negative** => real data has a length-scale
  signal; the plateau was a synthetic-`ls` artifact, and `ls` should be recoverable on real data.
- **Check 1 similar/narrower** *or* **Check 2 flat** => the plateau reflects the design geometry, not
  the synthetic generator; `ls` will be weakly identified on real data too — fix it or use a tight
  prior, or add test features at a range of distances from training to create the signal.